# H18b: Dense LR-sweep robustness check for the Muon-vs-SGD advantage

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/H18b_30X_FACTOR_LR_INDEPENDENCE/run_experiment.py`

This notebook is the analysis-and-presentation companion to the script above. In this first completion pass, the notebook no longer reimplements the experiment loop. Instead, it imports the script, calls its structured `run_experiment()` function, and builds tables/figures directly from the returned results.

## Truthful scope of this notebook

The current experiment is a **single-grid dense learning-rate sweep robustness check** in the toy deep-linear setting used elsewhere in the H18 series:

- SGD gets a 50-point log-spaced LR grid in `[1e-6, 0.5]`
- Muon gets a 50-point log-spaced LR grid in `[1e-5, 0.2]`
- Each LR is scored on 3 selection seeds by the median finite final loss
- The chosen best LR is then evaluated on 5 seeds
- The headline ratio is `SGD mean finite eval loss / Muon mean finite eval loss`

This notebook therefore **does not establish full LR-independence** across multiple LR-grid resolutions. It asks the narrower question: *what does the current toy experiment show when the same protocol is run with the denser 50-point LR sweeps already implemented in the script?*
            

## Notebook-safe import and path handling

The original notebook failed because it used script-style path resolution that is not defined in a normal notebook kernel. The version below resolves the repository root by walking upward from the current working directory until it finds the counterpart script path.

This makes the notebook explicit about where it is importing from while remaining notebook-safe.
            

In [ ]:
from pathlib import Path
import importlib.util
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

EXPERIMENT_RELATIVE_DIR = Path("experiments/Tier_1_Core_Mechanism_Tests/H18b_30X_FACTOR_LR_INDEPENDENCE")
SCRIPT_NAME = "run_experiment.py"


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / EXPERIMENT_RELATIVE_DIR / SCRIPT_NAME).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {EXPERIMENT_RELATIVE_DIR / SCRIPT_NAME} from cwd={start} or its parents."
    )


REPO_ROOT = find_repo_root()
SCRIPT_PATH = REPO_ROOT / EXPERIMENT_RELATIVE_DIR / SCRIPT_NAME

spec = importlib.util.spec_from_file_location("h18b_run_experiment", SCRIPT_PATH)
h18b = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(h18b)

print(f"Repo root: {REPO_ROOT}")
print(f"Counterpart script: {SCRIPT_PATH}")
            

## Reproducibility, runtime, and execution mode

By default, the notebook runs the script's **full default configuration**. On CPU, that may take several minutes because the script evaluates many `(depth, optimizer, LR, seed)` combinations.

For maintenance or smoke testing, this notebook also supports a small override via the environment variable `H18B_NOTEBOOK_SMOKE=1`. That path reduces depth count, LR-grid size, seed count, and training steps while preserving the same code path.
            

In [ ]:
NOTEBOOK_RUN_CONFIG = None
if os.environ.get("H18B_NOTEBOOK_SMOKE") == "1":
    NOTEBOOK_RUN_CONFIG = {
        "depths": [2],
        "sgd_lr_grid": np.logspace(-6, np.log10(0.5), 5).tolist(),
        "muon_lr_grid": np.logspace(-5, np.log10(0.2), 5).tolist(),
        "num_steps": 20,
        "num_seeds": 3,
        "selection_seed_count": 2,
    }

run_mode = "SMOKE" if NOTEBOOK_RUN_CONFIG is not None else "DEFAULT"
print(f"Notebook run mode: {run_mode}")

run_start = time.time()
RESULTS = h18b.run_experiment(config=NOTEBOOK_RUN_CONFIG, verbose=False)
NOTEBOOK_RUNTIME_SECONDS = time.time() - run_start

CONFIG = RESULTS["config"]
SUMMARY = RESULTS["summary"]
DEPTH_RESULTS = RESULTS["depth_results"]
DEPTH_MAP = {record["depth"]: record for record in DEPTH_RESULTS}

print(f"Notebook execution runtime: {NOTEBOOK_RUNTIME_SECONDS:.2f}s")
print(f"Experiment title: {RESULTS['title']}")
print(f"Experiment scope: {RESULTS['scope']}")
print(f"Primary metric caveat: {RESULTS['primary_metric_caveat']}")
            

## Reproducibility record and current configuration

The table below records the concrete script identity, scope, seed choices, LR grids, heuristic thresholds, and rough training-call count implied by the returned configuration.

The training-call count is a useful runtime proxy:

\[
	ext{total calls} = |	ext{depths}| 	imes \left((|	ext{SGD grid}| + |	ext{Muon grid}|) 	imes |	ext{selection seeds}| + 2 	imes |	ext{evaluation seeds}|ight).
\]
            

In [ ]:
training_call_count = len(CONFIG["depths"]) * (
    (len(CONFIG["sgd_lr_grid"]) + len(CONFIG["muon_lr_grid"])) * len(RESULTS["selection_seeds"])
    + 2 * len(RESULTS["evaluation_seeds"])
)

REPRO_DF = pd.DataFrame(
    [
        ("experiment_id", RESULTS["experiment_id"]),
        ("counterpart_script", str(SCRIPT_PATH.relative_to(REPO_ROOT))),
        ("scope", RESULTS["scope"]),
        ("run_mode", run_mode),
        ("notebook_runtime_seconds", f"{NOTEBOOK_RUNTIME_SECONDS:.2f}"),
        ("depths", RESULTS["config"]["depths"]),
        ("selection_seeds", RESULTS["selection_seeds"]),
        ("evaluation_seeds", RESULTS["evaluation_seeds"]),
        ("sgd_lr_grid_size", len(CONFIG["sgd_lr_grid"])),
        ("muon_lr_grid_size", len(CONFIG["muon_lr_grid"])),
        ("num_steps", CONFIG["num_steps"]),
        ("momentum", CONFIG["momentum"]),
        ("ns_iters", CONFIG["ns_iters"]),
        ("batch_size", CONFIG["batch_size"]),
        ("approx_training_calls", training_call_count),
        ("advantage_range_heuristic", tuple(CONFIG["advantage_range"])),
        ("cv_threshold_heuristic", CONFIG["cv_threshold"]),
        ("spearman_abs_threshold_heuristic", CONFIG["spearman_abs_threshold"]),
        ("primary_metric_caveat", RESULTS["primary_metric_caveat"]),
    ],
    columns=["field", "value"],
)

REPRO_DF
            

## Depthwise summary table and tidy raw-result frames

The next cell constructs three analysis tables directly from the script output:

1. `SUMMARY_DF`: one row per depth with best LR choices, evaluation losses, finite counts, and the depthwise advantage.
2. `LR_SWEEP_DF`: one row per `(depth, optimizer, LR)` with selection losses, finite counts, and median finite selection loss.
3. `EVAL_SEED_DF`: one row per `(depth, optimizer, seed)` with the final evaluation loss at the chosen best LR.

This makes divergence visible instead of burying it in a text printout.
            

In [ ]:
summary_rows = []
lr_rows = []
eval_rows = []

for depth_record in DEPTH_RESULTS:
    depth = depth_record["depth"]
    sgd = depth_record["optimizers"]["sgd"]
    muon = depth_record["optimizers"]["muon"]

    unstable_optimizers = [
        opt_name.upper()
        for opt_name, opt_result in depth_record["optimizers"].items()
        if not opt_result["evaluation_all_finite"]
    ]

    summary_rows.append(
        {
            "depth": depth,
            "sgd_best_lr": sgd["best_lr"],
            "sgd_selection_median_loss": sgd["best_selection_median_loss"],
            "sgd_eval_mean_loss": sgd["evaluation_mean_loss"],
            "sgd_eval_std_loss": sgd["evaluation_std_loss"],
            "sgd_eval_finite": f"{sgd['evaluation_finite_count']}/{CONFIG['num_seeds']}",
            "muon_best_lr": muon["best_lr"],
            "muon_selection_median_loss": muon["best_selection_median_loss"],
            "muon_eval_mean_loss": muon["evaluation_mean_loss"],
            "muon_eval_std_loss": muon["evaluation_std_loss"],
            "muon_eval_finite": f"{muon['evaluation_finite_count']}/{CONFIG['num_seeds']}",
            "advantage_sgd_over_muon": depth_record["advantage_sgd_over_muon"],
            "unstable_best_lr": ", ".join(unstable_optimizers) if unstable_optimizers else "none",
        }
    )

    for optimizer_name, optimizer_result in depth_record["optimizers"].items():
        for lr_entry in optimizer_result["lr_sweep"]:
            lr_rows.append(
                {
                    "depth": depth,
                    "optimizer": optimizer_name,
                    "lr": lr_entry["lr"],
                    "selection_seed_losses": lr_entry["selection_seed_losses"],
                    "selection_finite_count": lr_entry["selection_finite_count"],
                    "selection_median_finite_loss": lr_entry["selection_median_finite_loss"],
                    "is_best_lr": lr_entry["lr"] == optimizer_result["best_lr"],
                }
            )

        for seed, final_loss in zip(RESULTS["evaluation_seeds"], optimizer_result["evaluation_seed_losses"]):
            eval_rows.append(
                {
                    "depth": depth,
                    "optimizer": optimizer_name,
                    "seed": seed,
                    "best_lr": optimizer_result["best_lr"],
                    "final_loss": final_loss,
                    "is_finite": bool(np.isfinite(final_loss)),
                }
            )

SUMMARY_DF = pd.DataFrame(summary_rows).sort_values("depth").reset_index(drop=True)
LR_SWEEP_DF = pd.DataFrame(lr_rows).sort_values(["depth", "optimizer", "lr"]).reset_index(drop=True)
EVAL_SEED_DF = pd.DataFrame(eval_rows).sort_values(["depth", "optimizer", "seed"]).reset_index(drop=True)

SUMMARY_DF
            

## LR sweep selection landscapes

Each panel below shows the **selection-phase median finite final loss** as a function of learning rate for one `(depth, optimizer)` pair.

- The **red dashed line** marks the chosen best LR.
- **Orange markers** indicate LRs where not all selection seeds remained finite.
- **Black `x` markers** indicate LRs whose selection median was non-finite because every selection seed diverged.
- The panel title reports the **evaluation finite count at the chosen best LR** so that selection quality and evaluation stability can be read together.

These figures are the central sanity check for whether the best LR sits in a broad stable basin or near a fragile edge.
            

In [ ]:
depths = CONFIG["depths"]
selection_seed_total = len(RESULTS["selection_seeds"])

fig, axes = plt.subplots(len(depths), 2, figsize=(14, 3.2 * len(depths)), constrained_layout=True)
if len(depths) == 1:
    axes = np.array([axes])

for row_idx, depth in enumerate(depths):
    depth_record = DEPTH_MAP[depth]
    for col_idx, optimizer_name in enumerate(["sgd", "muon"]):
        ax = axes[row_idx, col_idx]
        optimizer_result = depth_record["optimizers"][optimizer_name]
        sweep_df = LR_SWEEP_DF[
            (LR_SWEEP_DF["depth"] == depth) & (LR_SWEEP_DF["optimizer"] == optimizer_name)
        ].copy()

        lrs = sweep_df["lr"].to_numpy(dtype=float)
        medians = sweep_df["selection_median_finite_loss"].to_numpy(dtype=float)
        finite_counts = sweep_df["selection_finite_count"].to_numpy(dtype=int)
        finite_mask = np.isfinite(medians)
        partial_mask = finite_mask & (finite_counts < selection_seed_total)
        all_diverged_mask = ~finite_mask

        if finite_mask.any():
            ax.semilogx(lrs[finite_mask], medians[finite_mask], marker="o", linewidth=1.5, label="finite selection median")
            if partial_mask.any():
                ax.scatter(
                    lrs[partial_mask],
                    medians[partial_mask],
                    color="orange",
                    edgecolor="black",
                    s=45,
                    zorder=3,
                    label="selection not fully finite",
                )
            ax.axvline(optimizer_result["best_lr"], color="red", linestyle="--", linewidth=1.2, label="chosen best LR")

            if all_diverged_mask.any():
                y_anchor = float(np.nanmax(medians[finite_mask]) * 1.35)
                ax.scatter(
                    lrs[all_diverged_mask],
                    np.full(all_diverged_mask.sum(), y_anchor),
                    marker="x",
                    color="black",
                    s=35,
                    label="all selection seeds diverged",
                )
                ymin, ymax = ax.get_ylim()
                ax.set_ylim(ymin, max(ymax, y_anchor * 1.12))
        else:
            ax.set_xscale("log")
            ax.text(0.5, 0.5, "No finite selection medians", transform=ax.transAxes, ha="center", va="center")

        ax.set_title(
            f"L={depth}, {optimizer_name.upper()} | eval finite "
            f"{optimizer_result['evaluation_finite_count']}/{CONFIG['num_seeds']}"
        )
        ax.set_xlabel("learning rate")
        ax.set_ylabel("median finite final loss")

handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=min(4, len(labels)), bbox_to_anchor=(0.5, 1.02))

plt.show()
            

## Final evaluation losses and depthwise advantage

The next figure summarizes what happens **after LR selection**:

- **Left:** mean finite evaluation loss at the chosen best LR, with standard-deviation error bars across finite seeds.
- **Right:** the depthwise advantage ratio `SGD / Muon` with the heuristic T1 band `[5x, 100x]` drawn for reference.
- Point annotations show finite evaluation counts so that a low mean based on incomplete finite runs is not mistaken for a fully stable result.
            

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), constrained_layout=True)

ax = axes[0]
ax.errorbar(
    SUMMARY_DF["depth"],
    SUMMARY_DF["sgd_eval_mean_loss"],
    yerr=SUMMARY_DF["sgd_eval_std_loss"],
    marker="o",
    linewidth=1.5,
    capsize=4,
    label="SGD",
)
ax.errorbar(
    SUMMARY_DF["depth"],
    SUMMARY_DF["muon_eval_mean_loss"],
    yerr=SUMMARY_DF["muon_eval_std_loss"],
    marker="s",
    linewidth=1.5,
    capsize=4,
    label="Muon",
)
ax.set_yscale("log")
ax.set_xlabel("depth")
ax.set_ylabel("mean finite evaluation loss")
ax.set_title("Evaluation losses at each optimizer's chosen best LR")
ax.legend()

for _, row in SUMMARY_DF.iterrows():
    ax.annotate(row["sgd_eval_finite"], (row["depth"], row["sgd_eval_mean_loss"]), textcoords="offset points", xytext=(5, 6), fontsize=8)
    ax.annotate(row["muon_eval_finite"], (row["depth"], row["muon_eval_mean_loss"]), textcoords="offset points", xytext=(5, -12), fontsize=8)

ax2 = axes[1]
ax2.plot(SUMMARY_DF["depth"], SUMMARY_DF["advantage_sgd_over_muon"], marker="o", linewidth=1.8, color="tab:purple")
ax2.axhspan(
    CONFIG["advantage_range"][0],
    CONFIG["advantage_range"][1],
    color="tab:green",
    alpha=0.12,
    label="T1 heuristic band",
)
ax2.axhline(CONFIG["advantage_range"][0], color="tab:green", linestyle=":", linewidth=1)
ax2.axhline(CONFIG["advantage_range"][1], color="tab:green", linestyle=":", linewidth=1)
if (SUMMARY_DF["advantage_sgd_over_muon"] > 0).all():
    ax2.set_yscale("log")
ax2.set_xlabel("depth")
ax2.set_ylabel("advantage = SGD mean loss / Muon mean loss")
ax2.set_title("Depthwise advantage with finite-count annotations")
ax2.legend()

for _, row in SUMMARY_DF.iterrows():
    annotation = f"SGD {row['sgd_eval_finite']} | Muon {row['muon_eval_finite']}"
    ax2.annotate(annotation, (row["depth"], row["advantage_sgd_over_muon"]), textcoords="offset points", xytext=(5, 6), fontsize=8)

plt.show()
            

## Global heuristic verdict and instability caveats

The script still reports the legacy T1/T2/T3 summary statistics for parity with the earlier experiment series, but the wording is now calibrated:

- T1/T2/T3 are **heuristic checks**, not formal hypothesis tests.
- The Spearman summary is reported as a descriptive monotonicity indicator only.
- If any chosen best LR does not evaluate cleanly on all seeds, that instability is called out explicitly below.
            

In [ ]:
VERDICT_DF = pd.DataFrame(
    [
        {
            "test": "T1_advantage_range",
            "rule": f"all advantages in ({CONFIG['advantage_range'][0]:.0f}x, {CONFIG['advantage_range'][1]:.0f}x)",
            "value": ", ".join(
                f"L={item['depth']}: {item['advantage_sgd_over_muon']:.2f}x"
                for item in SUMMARY["depth_advantages"]
            ),
            "pass": SUMMARY["pass_flags"]["T1_advantage_range"],
        },
        {
            "test": "T2_cv_log_advantage",
            "rule": f"CV(log advantage) < {CONFIG['cv_threshold']:.3f}",
            "value": SUMMARY["cv_log_advantage"],
            "pass": SUMMARY["pass_flags"]["T2_cv_log_advantage"],
        },
        {
            "test": "T3_no_monotone_depth_trend",
            "rule": f"|Spearman rho| < {CONFIG['spearman_abs_threshold']:.3f}",
            "value": SUMMARY["spearman_depth_advantage"],
            "pass": SUMMARY["pass_flags"]["T3_no_monotone_depth_trend"],
        },
    ]
)

print(f"Overall heuristic verdict: {'PASS' if SUMMARY['overall_pass'] else 'FAIL'}")
print("T3 caveat: the Spearman threshold is heuristic and is not a formal significance test.")
print()

if SUMMARY["unstable_settings"]:
    print("Chosen best LR settings with incomplete finite evaluation:")
    UNSTABLE_DF = pd.DataFrame(SUMMARY["unstable_settings"])
    print(UNSTABLE_DF.to_string(index=False))
else:
    UNSTABLE_DF = pd.DataFrame(columns=["depth", "optimizer", "best_lr", "evaluation_finite_count", "evaluation_total_count"])
    print("All chosen best LRs evaluated finitely on all seeds.")

VERDICT_DF
            

## Calibrated conclusion

This final cell deliberately states only what the current implementation actually measures. It is designed to track the returned metrics rather than a pre-written narrative.
            

In [ ]:
pass_items = [name for name, passed in SUMMARY["pass_flags"].items() if passed]
fail_items = [name for name, passed in SUMMARY["pass_flags"].items() if not passed]
advantage_text = ", ".join(
    f"L={item['depth']}: {item['advantage_sgd_over_muon']:.2f}x"
    for item in SUMMARY["depth_advantages"]
)

conclusion_lines = [
    "This notebook reproduces the script-defined dense 50-point LR-sweep experiment and analyzes the returned structured results rather than reimplementing the loop.",
    f"Observed depthwise advantages (SGD mean finite eval loss / Muon mean finite eval loss): {advantage_text}.",
    (
        f"Heuristic outcome: overall={'PASS' if SUMMARY['overall_pass'] else 'FAIL'}; "
        f"passed={pass_items if pass_items else 'none'}; failed={fail_items if fail_items else 'none'}."
    ),
    (
        f"Global summaries: mean advantage={SUMMARY['mean_advantage']:.2f}x, "
        f"geometric mean advantage={SUMMARY['geometric_mean_advantage']:.2f}x, "
        f"CV(log advantage)={SUMMARY['cv_log_advantage']:.3f}, "
        f"Spearman rho={SUMMARY['spearman_depth_advantage']:.3f}."
    ),
    "Interpretation boundary: these numbers come from a single dense LR grid per optimizer, so they support only a dense-grid robustness readout at this resolution, not a full LR-independence claim across grid granularities.",
    "Metric caveat: the historical primary metric averages only finite evaluation losses. Any optimizer-depth pair with an evaluation finite count below the total seed count should be treated as unstable at its selected best LR.",
    "A stronger claim about LR-independence would require extending the script to compare multiple LR-grid resolutions under the same selection/evaluation protocol.",
]

if SUMMARY["unstable_settings"]:
    unstable_text = "; ".join(
        f"depth={entry['depth']}, optimizer={entry['optimizer']}, best_lr={entry['best_lr']:.2e}, finite={entry['evaluation_finite_count']}/{entry['evaluation_total_count']}"
        for entry in SUMMARY["unstable_settings"]
    )
    conclusion_lines.append(f"Instability observed at chosen best LR(s): {unstable_text}.")
else:
    conclusion_lines.append("All chosen best LRs evaluated finitely on all available evaluation seeds.")

print((chr(10) * 2).join(conclusion_lines))
